# Designator Query Test with ROS and KnowRob

This notebook demonstrates how to send incremental designator queries to the `/knowrob/designator/query_incremental` action server in ROS, using the `DesignatorQueryIncrementalAction` from the `knowrob_designator` package. It covers two example queries:
1. **Breakfast query**: What to use for breakfast.
2. **Milk location query**: Where is the Milk stored?

> **Note:** Make sure that your ROS environment is sourced and that the `knowrob_designator` action server is running before executing these cells.

In [ ]:
# Imports and helper function definition
import rospy
import actionlib
import json
from knowrob_designator.msg import DesignatorQueryIncrementalAction, DesignatorQueryIncrementalGoal

def send_query(query_json, query_type="entityvar", query_id=""):
    """
    Send an incremental designator query to the KnowRob designator action server.
    
    Args:
        query_json (dict): The designator query expressed as a JSON-like dictionary.
        query_type (str): The type of query (default: "entityvar").
        query_id (str): An optional query ID for incremental queries (default: "").
    Returns:
        result: The result returned by the action server (an instance of DesignatorQueryIncrementalResult).
    """
    # Create action client for the designator query
    client = actionlib.SimpleActionClient(
        '/knowrob/designator/query_incremental',
        DesignatorQueryIncrementalAction
    )

    rospy.loginfo("Waiting for action server...")
    client.wait_for_server()
    rospy.loginfo("Action server available.")

    # Create and populate the goal message
    goal = DesignatorQueryIncrementalGoal()
    goal.query_type = query_type
    goal.query_id = query_id
    goal.designator_json = json.dumps(query_json)

    # Send goal and wait for result
    rospy.loginfo(f"Sending query (query_id={query_id}): {goal.designator_json}")
    client.send_goal(goal)
    client.wait_for_result()

    result = client.get_result()
    return result

In [ ]:
# Initialize ROS node
# Note: In a Jupyter notebook, make sure ROS_MASTER_URI and ROS environment are set up correctly.
rospy.init_node('designator_query_test', anonymous=True)

## Query 1: What to use for breakfast

Here we send a query to find any object that is used for breakfast. We perform it in two steps:
1. Send the initial query.
2. Send an incremental (empty) query using the returned `query_id` to retrieve remaining bindings.


In [ ]:
# Define the breakfast query
breakfast_query = {"anObject": {"type": "?x", "usedFor": "breakfast"}}

# Send the first part of the query
result1 = send_query(breakfast_query)
print("--- Breakfast query result 1 ---")
print(f"Success: {result1.success}")
print(f"Binding: {result1.binding_as_json}")
print(f"Query ID: {result1.query_id}")

# Send the incremental (empty) query to fetch more bindings if available
result2 = send_query(query_json={}, query_id=str(result1.query_id))
print("--- Breakfast query result 2 (incremental) ---")
print(f"Success: {result2.success}")
print(f"Binding: {result2.binding_as_json}")
print(f"Query ID: {result2.query_id}")

## Query 2: Where is the Milk stored?

This query searches for an action of type `searching` involving the object `Milk`,
and attempts to bind a location variable `?x` that satisfies the constraint of being inside some URDF link.


In [ ]:
# Define the milk location query
milk_location_query = {
    "anAction": {
        "type": "searching",
        "anObject": {"type": "Milk"},
        "aLocation": {
            "?x": {
                "insideOf": {
                    "anObject": {"URDFLink": "?_"}
                }
            }
        }
    }
}

# Send the milk location query
result3 = send_query(milk_location_query)
print("--- Milk location query result ---")
print(f"Success: {result3.success}")
print(f"Binding: {result3.binding_as_json}")
print(f"Query ID: {result3.query_id}")

### Notes
- Each call to `send_query` will block until the action server returns a result.
- The returned `binding_as_json` may contain one or more variable bindings in JSON format.
- If the result’s `success` field is `True`, it indicates that at least one binding was found.
- Incremental queries (empty `query_json` with a previous `query_id`) can be used to fetch additional bindings if they exist.